# 03 - Heart Failure Feasibility Analysis & Mortality Outcome Modeling
## Clinical Decision-Support Prototype: Research Outcome Modeling

> **MANDATORY CLINICAL & ARCHITECTURAL DIRECTIVE (Rule 8 & 20)**:
> 1. **Target Definition**: The target in this dataset is strictly **`DEATH_EVENT`** (all-cause mortality observed during follow-up).
> 2. **Clinical Population**: The study cohort consists exclusively of patients with **confirmed, severe heart failure (NYHA Class III-IV)**.
> 3. **Task Mismatch**: This dataset **does NOT support** and must **NEVER be used for** screening or diagnosing the onset of heart failure. There are no healthy controls.
> 4. **Leakage Safeguard**: Follow-up observation duration (`time`) is excluded from prospective inference to prevent look-ahead bias.


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay

PROJECT_ROOT = Path('.').resolve().parent.parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from Machine_Learning.src.data_loader import load_heart_failure_data
from Machine_Learning.src.preprocessing import (
    prepare_heart_failure_splits,
    preprocess_inference_input,
    HEART_FAILURE_RAW_FEATURES,
)
from Machine_Learning.src.train_heart_failure import (
    get_hf_candidate_models,
    train_and_select_best_hf_model,
)
from Machine_Learning.src.evaluate import evaluate_binary_classifier, print_evaluation_summary

sns.set_theme(style='whitegrid', palette='muted')
print('Environment initialized successfully. Project root:', PROJECT_ROOT)


Environment initialized successfully. Project root: C:\Users\lodha\Downloads\Heart_Disease_Screening


## 1. Clinical Target Mismatch & Intended Use Evaluation
### Why this dataset CANNOT diagnose Heart Failure
- In a true screening/diagnostic dataset, the negative class ($y=0$) represents individuals without the disease, and positive ($y=1$) represents individuals with the disease.
- In this dataset (Chicco & Jurman, 2020), **100% of the 299 patients had clinically diagnosed heart failure**.
- The binary target `DEATH_EVENT` indicates whether the patient died during subsequent follow-up ($96$ deaths, $203$ survivors).
- Therefore, the only valid task supported is **post-admission mortality risk stratification for confirmed heart failure patients**.


In [2]:
df_raw = load_heart_failure_data()
splits = prepare_heart_failure_splits(
    df_raw, test_size=0.2, val_size=0.1, random_state=42
)

print(f'Total Dataset Records: {len(df_raw)}')
print(f'Training fold       : {splits.X_train.shape[0]} samples ({round(splits.X_train.shape[0]/len(df_raw)*100, 1)}%)')
print(f'Validation fold     : {splits.X_val.shape[0]} samples ({round(splits.X_val.shape[0]/len(df_raw)*100, 1)}%)')
print(f'Held-out Test fold  : {splits.X_test.shape[0]} samples ({round(splits.X_test.shape[0]/len(df_raw)*100, 1)}%)')
print(f'Transformed Features: {splits.X_train.shape[1]}')
print(f'Mortality Rate in Train: {np.mean(splits.y_train):.3f}, Test: {np.mean(splits.y_test):.3f}')


2026-10-15 00:08:30,653 - Machine_Learning.src.data_loader - INFO - Loaded Heart Failure dataset: shape=(299, 13) from C:\Users\lodha\Downloads\Heart_Disease_Screening\Machine_Learning\datasets\raw\heart_failure.csv


Total Dataset Records: 299
Training fold       : 209 samples (69.9%)
Validation fold     : 30 samples (10.0%)
Held-out Test fold  : 60 samples (20.1%)
Transformed Features: 16
Mortality Rate in Train: 0.321, Test: 0.317


## 2. Baseline & Candidate Model Comparison
Evaluating five candidate classifiers on the validation fold ($n=30$):
1. **Dummy Baseline**: Stratified chance classifier
2. **Logistic Regression (Balanced)**: Regularized linear model with inverse class weighting
3. **Random Forest**: Shallow ensemble (100 trees, max depth 4) to control overfitting on $n=209$ training samples
4. **Gradient Boosting**: Shallow boosting (80 estimators, max depth 2)
5. **Support Vector Classifier (RBF)**: Non-linear kernel


In [3]:
best_name, best_model, val_results = train_and_select_best_hf_model(splits, random_state=42)

val_summary = []
for name, res in val_results.items():
    val_summary.append({
        'Model': name,
        'Val Accuracy': res['accuracy'],
        'Val Balanced Acc': res['balanced_accuracy'],
        'Val Sensitivity': res['recall_sensitivity'],
        'Val Specificity': res['specificity'],
        'Val Precision': res['precision'],
        'Val F1': res['f1_score'],
        'Val ROC-AUC': res.get('roc_auc'),
        'Val Brier': res.get('brier_score')
    })

val_df = pd.DataFrame(val_summary).sort_values(by='Val ROC-AUC', ascending=False)
print('Heart Failure Candidate Model Validation Comparison:')
print(val_df.to_string(index=False))


2026-10-15 00:08:30,940 - Machine_Learning.src.train_heart_failure - INFO - Evaluating 5 candidate models on Heart Failure validation fold...


2026-10-15 00:08:30,965 - Machine_Learning.src.train_heart_failure - INFO - Candidate [Dummy (Stratified Baseline)]: Val ROC-AUC=0.4000, Val F1=0.1176, Val Balanced Acc=0.4000


2026-10-15 00:08:31,007 - Machine_Learning.src.train_heart_failure - INFO - Candidate [Logistic Regression (Balanced)]: Val ROC-AUC=0.8800, Val F1=0.6957, Val Balanced Acc=0.7750


2026-10-15 00:08:31,329 - Machine_Learning.src.train_heart_failure - INFO - Candidate [Random Forest (Depth 4)]: Val ROC-AUC=0.8800, Val F1=0.6667, Val Balanced Acc=0.7500


2026-10-15 00:08:31,488 - Machine_Learning.src.train_heart_failure - INFO - Candidate [Gradient Boosting (Shallow)]: Val ROC-AUC=0.8900, Val F1=0.7368, Val Balanced Acc=0.8000


C:\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
2026-10-15 00:08:31,535 - Machine_Learning.src.train_heart_failure - INFO - Candidate [Support Vector Classifier (RBF)]: Val ROC-AUC=0.8300, Val F1=0.6364, Val Balanced Acc=0.7250


2026-10-15 00:08:31,536 - Machine_Learning.src.train_heart_failure - INFO - Selected top Heart Failure model: 'Gradient Boosting (Shallow)' (Validation Score: 0.8900)


Heart Failure Candidate Model Validation Comparison:
                          Model  Val Accuracy  Val Balanced Acc  Val Sensitivity  Val Specificity  Val Precision  Val F1  Val ROC-AUC  Val Brier
    Gradient Boosting (Shallow)        0.8333             0.800              0.7             0.90         0.7778  0.7368         0.89     0.1382
        Random Forest (Depth 4)        0.7667             0.750              0.7             0.80         0.6364  0.6667         0.88     0.1588
 Logistic Regression (Balanced)        0.7667             0.775              0.8             0.75         0.6154  0.6957         0.88     0.1512
Support Vector Classifier (RBF)        0.7333             0.725              0.7             0.75         0.5833  0.6364         0.83     0.1539
    Dummy (Stratified Baseline)        0.5000             0.400              0.1             0.70         0.1429  0.1176         0.40     0.5000


## 3. Final Evaluation on Held-Out Test Data
Evaluating the selected model (`Gradient Boosting (Shallow)`) on the unseen Test fold ($n=60$, with $19$ deaths and $41$ survivors).


In [4]:
y_test_pred = best_model.predict(splits.X_test)
y_test_prob = best_model.predict_proba(splits.X_test)[:, 1]

test_metrics = evaluate_binary_classifier(splits.y_test, y_test_pred, y_test_prob)
print_evaluation_summary(test_metrics, f'HELD-OUT TEST SET ({best_name})')



EVALUATION SUMMARY: HELD-OUT TEST SET (Gradient Boosting (Shallow))
Sample Count       : 60
Accuracy           : 0.7000
Balanced Accuracy  : 0.5969
Sensitivity/Recall : 0.3158
Specificity        : 0.8780
Precision          : 0.5455
F1 Score           : 0.4000
ROC-AUC            : 0.7766
PR-AUC             : 0.5241
Brier Score        : 0.1847

Confusion Matrix:
  TN: 36    FP: 5    
  FN: 13    TP: 6    



## 4. Diagnostic Performance Visualizations
Generating Confusion Matrix, ROC Curve, and Precision-Recall Curve on the held-out test set.


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion Matrix
cm = confusion_matrix(splits.y_test, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Survived', 'Deceased'])
disp.plot(ax=axes[0], cmap='Reds', colorbar=False)
axes[0].set_title(f'Confusion Matrix\n({best_name})')
axes[0].grid(False)

# 2. ROC Curve
fpr, tpr, _ = roc_curve(splits.y_test, y_test_prob)
axes[1].plot(fpr, tpr, color='crimson', lw=2, label=f'ROC-AUC = {test_metrics["roc_auc"]:.3f}')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[1].set_title('Receiver Operating Characteristic (ROC)')
axes[1].set_xlabel('False Positive Rate (1 - Specificity)')
axes[1].set_ylabel('True Positive Rate (Sensitivity)')
axes[1].legend(loc='lower right')

# 3. Precision-Recall Curve
prec, rec, _ = precision_recall_curve(splits.y_test, y_test_prob)
baseline_prevalence = np.mean(splits.y_test)
axes[2].plot(rec, prec, color='darkorange', lw=2, label=f'PR-AUC = {test_metrics["pr_auc"]:.3f}')
axes[2].axhline(baseline_prevalence, color='gray', linestyle='--', label=f'Baseline ({baseline_prevalence:.2f})')
axes[2].set_title('Precision-Recall Curve')
axes[2].set_xlabel('Recall (Sensitivity)')
axes[2].set_ylabel('Precision (Positive Predictive Value)')
axes[2].legend(loc='lower left')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'Machine_Learning' / 'notebooks' / 'hf_model_performance.png', dpi=150)
plt.close()
print('Saved diagnostic plot: hf_model_performance.png')


Saved diagnostic plot: hf_model_performance.png


## 5. Small-Sample Uncertainty & Clinical Feasibility Conclusion
1. **Severe Small-Sample Uncertainty**:
   - Total dataset contains only 299 patients and 96 deaths.
   - The test set contains only 19 deaths ($31.7\%$ test prevalence). A single patient outcome shifts sensitivity by $>5.2\%$.
   - The wide confidence intervals reflect substantial statistical variance inherent to small clinical cohorts.
2. **Leakage Safeguards vs Baseline Discriminability**:
   - Omitting `time` drops test sensitivity to $31.6\%$ while maintaining high specificity ($87.8\%$) and solid ROC-AUC ($0.7766$).
   - This drop reflects real-world clinical reality: predicting mortality months in advance from a single routine lab panel without knowing future follow-up duration is challenging.
3. **Final Architectural Status**:
   - **Supported Use**: Specialized exploratory research experiment for secondary risk stratification in patients with diagnosed heart failure.
   - **Explicit Unsupported Status**: **UNSUPPORTED** for outpatient heart failure screening or diagnosis. The application must display an explicit unsupported status for heart failure screening rather than using this model inappropriately.
